# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Uman-66/Flyrank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule

I will prioritize items that are **stale and have enough observed volume to make the issue worth investigating**.

The baseline score gives higher priority to:
- greater staleness, because stale items may need a refresh;
- greater observed volume, because higher-volume items offer more potential impact.

This is a **decision-support baseline**, not a prediction of future performance.

### Signal checks

Before using the signals in the rule, I check two observed signals:

1. **Staleness** — flag-linked signal behind FlyRank refresh/staleness logic.
2. **Volume** — signal associated with quick-win prioritization.

Each signal is bucketed using only observed values available at scoring time. I report the sample size (`n`) and give each signal a one-word verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.

### Reason codes

The rule outputs exactly one reason code per item:

- `STALE_HIGH_VOLUME` — item is both relatively stale and relatively high-volume.
- `STALE` — item is relatively stale but not high-volume.
- `HIGH_VOLUME` — item is relatively high-volume but not relatively stale.
- `BASELINE` — neither condition is met.

### Action labels

- `REFRESH_PRIORITY` — stale + high-volume.
- `REFRESH_REVIEW` — stale.
- `VOLUME_REVIEW` — high-volume.
- `MONITOR` — neither condition.

In [ ]:
# 1. Signal checks + rule setup
# Uses only observed/current fields.
# No labels, product flags, or future-window fields are used.

from pathlib import Path
import pandas as pd
import numpy as np
import re
import sys
import subprocess
import os

# If running in Google Colab, ensure repository is cloned and working directory is set
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Uman-66/Flyrank-Internship.git"
REPO_DIR = "Flyrank-Internship"

if IN_COLAB:
    if not (Path(REPO_DIR).exists() or Path("data/raw/content_refresh_anonymized.csv").exists()):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=False)
    if Path(REPO_DIR).exists() and not Path("data/raw/content_refresh_anonymized.csv").exists():
        os.chdir(REPO_DIR)

# Determine repository root
ROOT = Path.cwd().resolve()
for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / "data" / "raw").exists() or (candidate / "data").exists() or (candidate / ".git").exists():
        ROOT = candidate
        break

if (ROOT / "Flyrank-Internship" / "data").exists():
    ROOT = ROOT / "Flyrank-Internship"

DATA_DIRS = [
    ROOT / "data" / "raw",
    ROOT / "data",
    ROOT / "work" / "data",
    ROOT
]


# -----------------------------
# Helper functions
# -----------------------------
def norm(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")


def find_column(df, candidates):
    normalized = {norm(c): c for c in df.columns}

    # Exact normalized match
    for candidate in candidates:
        if norm(candidate) in normalized:
            return normalized[norm(candidate)]

    # Partial match
    for c in df.columns:
        nc = norm(c)
        for candidate in candidates:
            ncan = norm(candidate)
            if ncan in nc or nc in ncan:
                return c

    return None


# -----------------------------
# Find input files
# -----------------------------
files = []

for directory in DATA_DIRS:
    if directory.exists():
        files.extend(directory.glob("*.csv"))
        files.extend(directory.glob("*.parquet"))
        files.extend(directory.rglob("*.csv"))
        files.extend(directory.rglob("*.parquet"))

# Deduplicate files and exclude generated baseline output
seen = set()
unique_files = []
for p in files:
    resolved = p.resolve()
    if resolved not in seen and p.name != "baseline_action_score.csv" and not p.name.startswith("."):
        seen.add(resolved)
        unique_files.append(p)

files = unique_files

if not files:
    raise FileNotFoundError(
        "No CSV or Parquet input file was found. "
        "Run this notebook from the FlyRank repository."
    )

print("Candidate input files:")
for p in files:
    print(" -", p)


# -----------------------------
# Find dataset containing both signals
# -----------------------------
preferred = [
    p for p in files
    if any(
        word in p.name.lower()
        for word in [
            "content_refresh",
            "query",
            "content",
            "page",
            "keyword",
            "item",
            "rank",
            "dataset"
        ]
    )
]

candidate_files = preferred if preferred else files

df = None
source_file = None
staleness_col = None
volume_col = None

for path in candidate_files:

    try:
        if path.suffix.lower() == ".parquet":
            test = pd.read_parquet(path)
        else:
            test = pd.read_csv(path)

        possible_staleness = find_column(
            test,
            [
                "content_age_days",
                "days_since_last_update",
                "staleness",
                "stale_days",
                "days_stale",
                "age_days",
                "days_since_update"
            ]
        )

        possible_volume = find_column(
            test,
            [
                "search_volume",
                "impressions_90d",
                "volume",
                "monthly_volume",
                "impressions",
                "impression_volume"
            ]
        )

        if possible_staleness is not None and possible_volume is not None:
            df = test.copy()
            source_file = path
            staleness_col = possible_staleness
            volume_col = possible_volume
            break

    except Exception as e:
        print(f"Could not read {path.name}: {e}")


# -----------------------------
# Validate dataset
# -----------------------------
if df is None:
    raise ValueError(
        "Could not find a dataset containing both a "
        "staleness-like column and a volume-like column.\n\n"
        "Available files:\n"
        + "\n".join(str(p) for p in candidate_files)
    )

print("\nSelected input file:", source_file)
print("Rows:", len(df))
print("Staleness column:", staleness_col)
print("Volume column:", volume_col)


# -----------------------------
# Prepare observed signals
# -----------------------------
work = df[
    [staleness_col, volume_col]
].copy()

work[staleness_col] = pd.to_numeric(
    work[staleness_col],
    errors="coerce"
)

work[volume_col] = pd.to_numeric(
    work[volume_col],
    errors="coerce"
)

work = work.dropna().copy()

if len(work) < 20:
    raise ValueError(
        f"Only {len(work)} usable rows remain. "
        "At least 20 rows are needed for the review."
    )


# -----------------------------
# Observed thresholds
# -----------------------------
stale_threshold = work[staleness_col].median()
volume_threshold = work[volume_col].median()

work["stale_bucket"] = np.where(
    work[staleness_col] >= stale_threshold,
    "HIGH",
    "LOW"
)

work["volume_bucket"] = np.where(
    work[volume_col] >= volume_threshold,
    "HIGH",
    "LOW"
)


# ============================================================
# SIGNAL 1 — STALENESS
# ============================================================

stale_table = (
    work
    .groupby("stale_bucket", observed=True)
    .agg(
        n=(staleness_col, "size"),
        mean_staleness=(staleness_col, "mean"),
        median_staleness=(staleness_col, "median"),
        mean_volume=(volume_col, "mean")
    )
    .reset_index()
)

print("\n" + "=" * 60)
print("SIGNAL 1 — STALENESS")
print("=" * 60)

print(stale_table.to_string(index=False))
print("\nn =", len(work))

stale_means = stale_table.set_index("stale_bucket")["mean_volume"]

high_stale_volume = stale_means.get("HIGH", np.nan)
low_stale_volume = stale_means.get("LOW", np.nan)

if pd.isna(high_stale_volume) or pd.isna(low_stale_volume):
    stale_verdict = "MIXED"
elif high_stale_volume > low_stale_volume:
    stale_verdict = "CONFIRMED"
elif high_stale_volume < low_stale_volume:
    stale_verdict = "OPPOSITE"
else:
    stale_verdict = "MIXED"

print("\nVerdict:", stale_verdict)

print(
    "Interpretation: observed bucket comparison only; "
    "this does not establish causality."
)


# ============================================================
# SIGNAL 2 — VOLUME
# ============================================================

volume_table = (
    work
    .groupby("volume_bucket", observed=True)
    .agg(
        n=(volume_col, "size"),
        mean_volume=(volume_col, "mean"),
        median_volume=(volume_col, "median"),
        mean_staleness=(staleness_col, "mean")
    )
    .reset_index()
)

print("\n" + "=" * 60)
print("SIGNAL 2 — VOLUME")
print("=" * 60)

print(volume_table.to_string(index=False))
print("\nn =", len(work))

volume_means = volume_table.set_index("volume_bucket")["mean_staleness"]

high_volume_staleness = volume_means.get("HIGH", np.nan)
low_volume_staleness = volume_means.get("LOW", np.nan)

if pd.isna(high_volume_staleness) or pd.isna(low_volume_staleness):
    volume_verdict = "MIXED"
elif high_volume_staleness > low_volume_staleness:
    volume_verdict = "CONFIRMED"
elif high_volume_staleness < low_volume_staleness:
    volume_verdict = "OPPOSITE"
else:
    volume_verdict = "MIXED"

print("\nVerdict:", volume_verdict)

print(
    "Interpretation: observed bucket comparison only; "
    "this does not establish causality."
)


# -----------------------------
# Rule thresholds
# -----------------------------
print("\n" + "=" * 60)
print("RULE THRESHOLDS")
print("=" * 60)

print("Staleness threshold:", stale_threshold)
print("Volume threshold:", volume_threshold)


## 2. Build the ranked queue

The baseline score combines two observed signals:

- normalized staleness
- normalized volume

Both are converted to percentile ranks so that their scales are comparable.

The final score is:

**baseline score = 0.60 × staleness percentile + 0.40 × volume percentile**

The score is used only to prioritize work. It is not a prediction or outcome label.

Each row receives exactly one reason code and one action label.

The resulting ranked queue is written to:

`work/outputs/baseline_action_score.csv`

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 2. Build the ranked queue and write the required CSV

from pathlib import Path
import numpy as np
import pandas as pd

queue = df.copy()

queue[staleness_col] = pd.to_numeric(queue[staleness_col], errors="coerce")
queue[volume_col] = pd.to_numeric(queue[volume_col], errors="coerce")

queue = queue.dropna(
    subset=[staleness_col, volume_col]
).copy()

# Percentile ranks use only observed values in this scoring dataset.
queue["staleness_pct"] = (
    queue[staleness_col]
    .rank(method="average", pct=True)
)

queue["volume_pct"] = (
    queue[volume_col]
    .rank(method="average", pct=True)
)

# One simple, fixed baseline score.
queue["score"] = (
    0.60 * queue["staleness_pct"]
    + 0.40 * queue["volume_pct"]
)

# Reason code: exactly ONE per row.
queue["reason_code"] = np.select(
    [
        (queue[staleness_col] >= stale_threshold) &
        (queue[volume_col] >= volume_threshold),

        queue[staleness_col] >= stale_threshold,

        queue[volume_col] >= volume_threshold
    ],
    [
        "STALE_HIGH_VOLUME",
        "STALE",
        "HIGH_VOLUME"
    ],
    default="BASELINE"
)

# Action label: exactly ONE per row.
queue["action"] = np.select(
    [
        queue["reason_code"].eq("STALE_HIGH_VOLUME"),
        queue["reason_code"].eq("STALE"),
        queue["reason_code"].eq("HIGH_VOLUME")
    ],
    [
        "REFRESH_PRIORITY",
        "REFRESH_REVIEW",
        "VOLUME_REVIEW"
    ],
    default="MONITOR"
)

# Rank highest score first.
queue = (
    queue
    .sort_values(
        ["score", staleness_col, volume_col],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

# Keep a clean output.
# Include an existing identifier if one is available.
id_col = find_column(
    queue,
    ["content_id", "query_id", "keyword_id", "item_id", "page_id", "id"]
)

output_columns = []

if id_col:
    output_columns.append(id_col)

output_columns += [
    "rank",
    "score",
    "reason_code",
    "action",
    staleness_col,
    volume_col
]

output_columns = list(dict.fromkeys(output_columns))

baseline_output = queue[output_columns].copy()

# Required output path.
output_dir = ROOT / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

baseline_output.to_csv(output_path, index=False)

print("Rows ranked:", len(baseline_output))
print("Output written:", output_path)
print("\nReason-code counts:")
print(baseline_output["reason_code"].value_counts())

print("\nAction counts:")
print(baseline_output["action"].value_counts())

print("\nTop 10:")
print(baseline_output.head(10).to_string(index=False))

assert len(baseline_output) == len(queue)
assert baseline_output["rank"].is_monotonic_increasing
assert baseline_output["reason_code"].notna().all()
assert baseline_output["action"].notna().all()
assert output_path.exists()

print("\nBaseline queue checks passed.")


## 3. Top-20 review

The following review looks at the highest-ranked 20 items from the baseline queue.

For every item I record:

- the action;
- the reason code;
- a confidence note based only on the observed signals;
- what would make the recommendation wrong.

The last point is important because this baseline is decision-support, not ground truth. A high score does not prove that an item needs the recommended action.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3. Top-20 review
# Generates one review row for each of the top 20 ranked items.

top20 = queue.head(20).copy()

def confidence_note(row):
    stale_strength = (
        (row[staleness_col] - stale_threshold)
        / (abs(stale_threshold) + 1e-9)
    )

    volume_strength = (
        (row[volume_col] - volume_threshold)
        / (abs(volume_threshold) + 1e-9)
    )

    if stale_strength > 0.50 and volume_strength > 0.50:
        return "High relative to both observed rule signals"
    elif stale_strength > 0:
        return "Higher relative staleness; volume is less supportive"
    elif volume_strength > 0:
        return "Higher relative volume; staleness is less supportive"
    else:
        return "Near the lower side of the rule thresholds"

def wrong_if(row):
    if row["reason_code"] == "STALE_HIGH_VOLUME":
        return (
            "Wrong if the observed staleness reflects an acceptable "
            "content state or the volume is not meaningful for this item."
        )

    if row["reason_code"] == "STALE":
        return (
            "Wrong if the item is intentionally unchanged, or if staleness "
            "does not indicate a useful refresh opportunity."
        )

    if row["reason_code"] == "HIGH_VOLUME":
        return (
            "Wrong if high volume does not translate into a useful "
            "decision opportunity for this item."
        )

    return (
        "Wrong if an important signal is missing from this simple baseline "
        "or if neither observed threshold is appropriate."
    )

review_rows = []

for _, row in top20.iterrows():
    review_rows.append({
        "rank": int(row["rank"]),
        "action": row["action"],
        "reason_code": row["reason_code"],
        "score": round(float(row["score"]), 4),
        "confidence_note": confidence_note(row),
        "what_would_make_it_wrong": wrong_if(row)
    })

top20_review = pd.DataFrame(review_rows)

print(top20_review.to_string(index=False))

assert len(top20_review) == min(20, len(queue))
assert top20_review["what_would_make_it_wrong"].notna().all()
assert top20_review["action"].notna().all()
assert top20_review["reason_code"].notna().all()

## 4. Weak picks + leakage check

### Weak picks

I will inspect the bottom of the ranked queue for cases that look questionable.

A weak pick is not automatically a failure. It is useful evidence about where this simple baseline may be over-simplifying the prioritization problem.

### Leakage check

The baseline uses only the two observed signals:

- staleness
- volume

It does not use:

- future performance;
- outcome/label columns;
- product flags;
- post-period measurements;
- manually assigned FlyRank flags as model inputs.

The FlyRank flag connection is used only to motivate the signal audit, not to create the score.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 4. Weak picks + leakage check

# Show the lowest-ranked 10 items.
weak_picks = queue.tail(10).copy()

weak_output = weak_picks[
    [
        "rank",
        "score",
        "reason_code",
        "action",
        staleness_col,
        volume_col
    ]
].copy()

print("WEAK PICKS — LOWEST 10")
print(weak_output.to_string(index=False))

print("\nWeak-pick interpretation:")
print(
    "These rows have the lowest baseline scores. They may still deserve "
    "attention if information outside this simple baseline matters."
)

# ---------- leakage checks ----------

# Explicitly identify suspicious feature names.
# These checks are intentionally conservative.
forbidden_patterns = [
    "label",
    "target",
    "outcome",
    "future",
    "post_period",
    "product_flag",
    "flyrank_flag",
    "flag"
]

feature_names = {
    norm(c)
    for c in queue.columns
}

# The original source data may contain columns we do not use.
# Therefore inspect ONLY the columns actually used by the baseline.
used_features = {
    norm(staleness_col),
    norm(volume_col)
}

leakage_hits = []

for feature in used_features:
    for forbidden in forbidden_patterns:
        if forbidden in feature:
            leakage_hits.append((feature, forbidden))

print("\nLEAKAGE CHECK")
print("Features used by baseline:", [staleness_col, volume_col])

if leakage_hits:
    raise AssertionError(
        f"Potential leakage-like feature name detected: {leakage_hits}"
    )

# Ensure the score was not created from label/flag columns.
score_dependencies = {
    "staleness": staleness_col,
    "volume": volume_col
}

print("Score dependencies:", score_dependencies)
print("No future-window, label-derived, or product-flag input is used by the score.")

# Check that reason/action are generated from the two signals only.
assert "score" in queue.columns
assert "reason_code" in queue.columns
assert "action" in queue.columns

print("\nLeakage checks passed.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.